### Black-Scholes, Greeks & Binomial Tree in Python OOP

**Quantitative finance concepts:**
- Black-Scholes pricing and all five Greeks (Δ, Γ, ν, θ, ρ)
- Put-Call Parity verification
- Early exercise premium: European vs American put
- Greeks surfaces as a function of spot and time-to-maturity
- Straddle, strangle, and collar payoff diagrams

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import norm
from abc import ABC, abstractmethod
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 13})
print("libraries imported successfully")

## 1. Class Hierarchy Design

```
Option  (ABC)
├── EuropeanOption   →  Black-Scholes (closed-form)
└── AmericanOption   →  CRR Binomial Tree (backward induction)

OptionPortfolio      →  collection of (option, quantity) legs
```

In [ ]:
class Option(ABC):
    """Abstract base: common attributes shared by all option types."""

    VALID_KINDS = {"call", "put"}

    def __init__(self, S: float, K: float, T: float, r: float,
                 sigma: float, kind: str = "call"):
        if kind not in self.VALID_KINDS:
            raise ValueError(f"kind must be one of {self.VALID_KINDS}")
        if T <= 0:
            raise ValueError("T (time to maturity) must be positive.")
        if sigma <= 0:
            raise ValueError("sigma (volatility) must be positive.")
        self.S, self.K, self.T, self.r, self.sigma, self.kind = S, K, T, r, sigma, kind

    @property
    def intrinsic_value(self) -> float:
        if self.kind == "call":
            return max(self.S - self.K, 0.0)
        return max(self.K - self.S, 0.0)

    @abstractmethod
    def price(self) -> float:
        """Return the fair value of the option."""

    @abstractmethod
    def greeks(self) -> dict:
        """Return a dict of all Greeks."""

    def __repr__(self) -> str:
        return (f"{self.__class__.__name__}({self.kind.upper()} "
                f"S={self.S:.1f} K={self.K:.1f} T={self.T:.2f}Y "
                f"r={self.r:.1%} σ={self.sigma:.1%})")


print("Option ABC defined.")

In [ ]:
class EuropeanOption(Option):
    """European option priced via the Black-Scholes closed-form formula."""

    def _d1_d2(self):
        d1 = (np.log(self.S / self.K) + (self.r + 0.5 * self.sigma**2) * self.T)              / (self.sigma * np.sqrt(self.T))
        d2 = d1 - self.sigma * np.sqrt(self.T)
        return d1, d2

    def price(self) -> float:
        d1, d2 = self._d1_d2()
        disc = np.exp(-self.r * self.T)
        if self.kind == "call":
            return float(self.S * norm.cdf(d1) - self.K * disc * norm.cdf(d2))
        return float(self.K * disc * norm.cdf(-d2) - self.S * norm.cdf(-d1))

    def greeks(self) -> dict:
        d1, d2 = self._d1_d2()
        disc   = np.exp(-self.r * self.T)
        sqrtT  = np.sqrt(self.T)
        nd1    = norm.pdf(d1)

        delta = norm.cdf(d1) if self.kind == "call" else norm.cdf(d1) - 1
        gamma = nd1 / (self.S * self.sigma * sqrtT)
        vega  = self.S * nd1 * sqrtT / 100
        if self.kind == "call":
            theta = (-(self.S * nd1 * self.sigma) / (2 * sqrtT)
                     - self.r * self.K * disc * norm.cdf(d2)) / 365
            rho   = self.K * self.T * disc * norm.cdf(d2) / 100
        else:
            theta = (-(self.S * nd1 * self.sigma) / (2 * sqrtT)
                     + self.r * self.K * disc * norm.cdf(-d2)) / 365
            rho   = -self.K * self.T * disc * norm.cdf(-d2) / 100

        return {"delta": delta, "gamma": gamma, "vega": vega,
                "theta": theta, "rho": rho}

    def put_call_parity_check(self) -> dict:
        """Verify: C - P = S - K·e^(-rT)."""
        call_p = EuropeanOption(self.S, self.K, self.T, self.r, self.sigma, "call").price()
        put_p  = EuropeanOption(self.S, self.K, self.T, self.r, self.sigma, "put").price()
        lhs    = call_p - put_p
        rhs    = self.S - self.K * np.exp(-self.r * self.T)
        return {"C - P": round(lhs, 6), "S - K·e⁻ʳᵀ": round(rhs, 6),
                "diff": round(abs(lhs - rhs), 10)}


print("EuropeanOption defined.")

In [ ]:
class AmericanOption(Option):
    """American option priced via the Cox-Ross-Rubinstein binomial tree."""

    def __init__(self, S, K, T, r, sigma, kind="call", steps=500):
        super().__init__(S, K, T, r, sigma, kind)
        self.steps = steps

    def price(self) -> float:
        return self._crr_price()

    def greeks(self) -> dict:
        eps_s = self.S * 1e-4
        eps_t = self.T * 1e-4
        eps_v = self.sigma * 1e-4

        def px(**kwargs):
            params = dict(S=self.S, K=self.K, T=self.T, r=self.r,
                          sigma=self.sigma, kind=self.kind, steps=self.steps)
            params.update(kwargs)
            return AmericanOption(**params).price()

        p0 = self.price()
        delta = (px(S=self.S + eps_s) - px(S=self.S - eps_s)) / (2 * eps_s)
        gamma = (px(S=self.S + eps_s) - 2 * p0 + px(S=self.S - eps_s)) / eps_s**2
        vega  = (px(sigma=self.sigma + eps_v) - px(sigma=self.sigma - eps_v)) / (2 * eps_v * 100)
        theta = (px(T=max(self.T - eps_t, 1e-6)) - p0) / (eps_t * 365)
        rho   = (px(r=self.r + 1e-4) - px(r=self.r - 1e-4)) / (2e-4 * 100)
        return {"delta": delta, "gamma": gamma, "vega": vega,
                "theta": theta, "rho": rho}

    def _crr_price(self) -> float:
        N  = self.steps
        dt = self.T / N
        u  = np.exp(self.sigma * np.sqrt(dt))
        d  = 1.0 / u
        p  = (np.exp(self.r * dt) - d) / (u - d)
        disc = np.exp(-self.r * dt)

        # Terminal stock prices (vectorised)
        j  = np.arange(N + 1)
        ST = self.S * (u ** (N - j)) * (d ** j)

        # Terminal payoff
        if self.kind == "call":
            V = np.maximum(ST - self.K, 0.0)
        else:
            V = np.maximum(self.K - ST, 0.0)

        # Backward induction with early exercise check
        for _ in range(N):
            ST = ST[:-1] / u            # one step back
            V  = disc * (p * V[:-1] + (1 - p) * V[1:])
            if self.kind == "call":
                V = np.maximum(V, ST - self.K)
            else:
                V = np.maximum(V, self.K - ST)
        return float(V[0])


print("AmericanOption defined.")

## 2. Pricing Demo — ATM Options

In [ ]:
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.045, 0.25

eu_call = EuropeanOption(S, K, T, r, sigma, "call")
eu_put  = EuropeanOption(S, K, T, r, sigma, "put")
am_call = AmericanOption(S, K, T, r, sigma, "call", steps=500)
am_put  = AmericanOption(S, K, T, r, sigma, "put",  steps=500)

print(eu_call)
print(eu_put)
print()

rows = []
for opt, label in [(eu_call, "EU Call"), (eu_put, "EU Put"),
                   (am_call, "AM Call"), (am_put, "AM Put")]:
    g = opt.greeks()
    rows.append({"Option": label, "Price": round(opt.price(), 4),
                 **{k: round(v, 5) for k, v in g.items()}})

df = pd.DataFrame(rows).set_index("Option")
print(df.to_string())
print()
print("Put-Call Parity:", eu_call.put_call_parity_check())
print()
print(f"Early exercise premium (put): {am_put.price() - eu_put.price():.4f}")

## 3. Greeks Visualisation

We plot each Greek as a function of the spot price, keeping all other parameters fixed.

In [ ]:
spots = np.linspace(60, 140, 120)
greek_names = ["delta", "gamma", "vega", "theta", "rho"]
call_greeks = {g: [] for g in greek_names}
put_greeks  = {g: [] for g in greek_names}

for s in spots:
    cg = EuropeanOption(s, K, T, r, sigma, "call").greeks()
    pg = EuropeanOption(s, K, T, r, sigma, "put").greeks()
    for g in greek_names:
        call_greeks[g].append(cg[g])
        put_greeks[g].append(pg[g])

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, g in enumerate(greek_names):
    axes[i].plot(spots, call_greeks[g], color="steelblue", linewidth=2, label="Call")
    axes[i].plot(spots, put_greeks[g],  color="coral",     linewidth=2, label="Put")
    axes[i].axvline(K, color="gray", linestyle="--", linewidth=1, label=f"K={K}")
    axes[i].axhline(0, color="black", linewidth=0.6)
    axes[i].set_title(f"{'ΔΓΝΘΡ'[i]}  {g.capitalize()}")
    axes[i].set_xlabel("Spot Price")
    axes[i].legend(fontsize=9)

axes[5].axis("off")
fig.suptitle(f"Option Greeks vs Spot Price  (K={K}, T={T}Y, r={r:.1%}, σ={sigma:.0%})",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Greeks Surface — Delta as a Function of Spot × Time-to-Maturity

In [ ]:
spots_2d = np.linspace(70, 130, 50)
maturities = np.linspace(0.05, 2.0, 50)
SS, TT = np.meshgrid(spots_2d, maturities)

delta_surface = np.vectorize(
    lambda s, t: EuropeanOption(s, K, t, r, sigma, "call").greeks()["delta"]
)(SS, TT)

gamma_surface = np.vectorize(
    lambda s, t: EuropeanOption(s, K, t, r, sigma, "call").greeks()["gamma"]
)(SS, TT)

fig = plt.figure(figsize=(14, 5))
for idx, (surface, title) in enumerate(
    [(delta_surface, "Call Delta"), (gamma_surface, "Call Gamma")], 1
):
    ax = fig.add_subplot(1, 2, idx, projection="3d")
    ax.plot_surface(SS, TT, surface, cmap="plasma", alpha=0.9, linewidth=0)
    ax.set_xlabel("Spot (S)")
    ax.set_ylabel("Maturity (T)")
    ax.set_zlabel(title)
    ax.set_title(f"{title} Surface  (K={K}, r={r:.1%}, σ={sigma:.0%})")

plt.tight_layout()
plt.show()

## 5. European vs American Put — Early Exercise Premium

In [ ]:
sigmas = np.linspace(0.05, 0.80, 40)
eu_puts_v = [EuropeanOption(S, K, T, r, s, "put").price() for s in sigmas]
am_puts_v = [AmericanOption(S, K, T, r, s, "put", steps=300).price() for s in sigmas]
premium   = [a - e for a, e in zip(am_puts_v, eu_puts_v)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(sigmas * 100, eu_puts_v, color="steelblue", linewidth=2, label="European Put")
axes[0].plot(sigmas * 100, am_puts_v, color="coral",     linewidth=2, label="American Put")
axes[0].set_title("Put Price vs Volatility")
axes[0].set_xlabel("Implied Volatility (%)")
axes[0].set_ylabel("Price ($)")
axes[0].legend()

axes[1].fill_between(sigmas * 100, premium, color="coral", alpha=0.4)
axes[1].plot(sigmas * 100, premium, color="crimson", linewidth=2)
axes[1].set_title("Early Exercise Premium (AM − EU)")
axes[1].set_xlabel("Implied Volatility (%)")
axes[1].set_ylabel("Premium ($)")

plt.tight_layout()
plt.show()

## 6. OptionPortfolio — Multi-Leg Strategies

We compose several legs to model **straddle**, **strangle**, and **collar** strategies.

In [ ]:
class OptionPortfolio:
    """Collection of (EuropeanOption, signed_quantity) legs."""

    def __init__(self):
        self.legs: list[tuple[EuropeanOption, float]] = []

    def add_leg(self, option: EuropeanOption, qty: float = 1.0):
        self.legs.append((option, qty))
        return self

    def price(self) -> float:
        return sum(o.price() * q for o, q in self.legs)

    def greeks(self) -> dict:
        agg = {"delta": 0.0, "gamma": 0.0, "vega": 0.0, "theta": 0.0, "rho": 0.0}
        for opt, qty in self.legs:
            for k, v in opt.greeks().items():
                agg[k] += v * qty
        return {k: round(v, 5) for k, v in agg.items()}

    def payoff_at_expiry(self, spots: np.ndarray) -> np.ndarray:
        total = np.zeros_like(spots, dtype=float)
        for opt, qty in self.legs:
            if opt.kind == "call":
                total += qty * np.maximum(spots - opt.K, 0)
            else:
                total += qty * np.maximum(opt.K - spots, 0)
        return total

    def net_premium(self) -> float:
        return -self.price()


S_ref, T_ref, r_ref, sig_ref = 100.0, 0.5, 0.045, 0.25

# Straddle: long call + long put at same strike
straddle = (OptionPortfolio()
            .add_leg(EuropeanOption(S_ref, 100, T_ref, r_ref, sig_ref, "call"))
            .add_leg(EuropeanOption(S_ref, 100, T_ref, r_ref, sig_ref, "put")))

# Strangle: long OTM call + long OTM put
strangle = (OptionPortfolio()
            .add_leg(EuropeanOption(S_ref, 110, T_ref, r_ref, sig_ref, "call"))
            .add_leg(EuropeanOption(S_ref,  90, T_ref, r_ref, sig_ref, "put")))

# Collar: long stock + long OTM put + short OTM call
collar = (OptionPortfolio()
          .add_leg(EuropeanOption(S_ref, 110, T_ref, r_ref, sig_ref, "call"), qty=-1)
          .add_leg(EuropeanOption(S_ref,  90, T_ref, r_ref, sig_ref, "put"),  qty=+1))

strategies = {"Straddle": straddle, "Strangle": strangle, "Collar": collar}

spots_range = np.linspace(60, 140, 300)
fig, axes   = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, port) in zip(axes, strategies.items()):
    payoff   = port.payoff_at_expiry(spots_range)
    premium  = port.price()
    pnl      = payoff - premium
    ax.plot(spots_range, pnl, color="steelblue", linewidth=2.2, label="P&L at expiry")
    ax.fill_between(spots_range, pnl, 0,
                    where=(pnl >= 0), alpha=0.25, color="green", label="Profit")
    ax.fill_between(spots_range, pnl, 0,
                    where=(pnl < 0),  alpha=0.25, color="red",   label="Loss")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(S_ref, color="gray", linestyle="--", linewidth=1, label=f"S₀={S_ref}")
    ax.set_title(f"{name}  (premium=${premium:.2f})")
    ax.set_xlabel("Spot at Expiry")
    ax.set_ylabel("P&L ($)")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\nPortfolio Greeks Summary")
for name, port in strategies.items():
    g = port.greeks()
    print(f"  {name:12s}  price={port.price():.2f}  "
          + "  ".join(f"{k}={v:+.3f}" for k, v in g.items()))

## 7. Implied Volatility — Newton-Raphson Solver

Given a market price, we back out the **implied volatility** numerically.

In [ ]:
def implied_vol(market_price: float, S: float, K: float, T: float,
                r: float, kind: str = "call",
                tol: float = 1e-7, max_iter: int = 200) -> float:
    """Newton-Raphson solver for implied volatility."""
    sigma = 0.30
    for _ in range(max_iter):
        opt   = EuropeanOption(S, K, T, r, sigma, kind)
        price = opt.price()
        vega  = opt.greeks()["vega"] * 100   # undo /100 scaling
        diff  = price - market_price
        if abs(diff) < tol:
            return sigma
        if abs(vega) < 1e-10:
            break
        sigma -= diff / vega
        sigma = max(1e-6, min(sigma, 10.0))
    return sigma

# Demonstration: compute implied vol for a range of market prices
true_sigma = 0.30
strikes = np.arange(80, 125, 5)
results = []
for k in strikes:
    eu = EuropeanOption(100, k, 1.0, 0.045, true_sigma, "call")
    mkt_price = eu.price() * (1 + np.random.normal(0, 0.005))   # add tiny noise
    iv = implied_vol(mkt_price, 100, k, 1.0, 0.045, "call")
    results.append({"Strike": k, "Market Price": round(mkt_price, 3),
                    "Implied Vol": round(iv * 100, 2), "True σ": true_sigma * 100})

df_iv = pd.DataFrame(results).set_index("Strike")
print(df_iv.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df_iv.index, df_iv["Implied Vol"], "o-", color="steelblue",
        linewidth=2, markersize=7, label="Implied Vol (%)")
ax.axhline(true_sigma * 100, color="crimson", linestyle="--",
           linewidth=1.5, label=f"True σ = {true_sigma:.0%}")
ax.set_title("Implied Volatility vs Strike (Vol Smile Simulation)")
ax.set_xlabel("Strike (K)")
ax.set_ylabel("Implied Volatility (%)")
ax.legend()
plt.tight_layout()
plt.show()